# Exploración de chunking — Código Nacional de Procedimientos Penales

Este notebook explora el documento antes de decidir la estrategia de chunking.

**Objetivos**
1. Cargar el Markdown.
2. Explorar `LIBRO`, `TÍTULO`, `CAPÍTULO` y `Artículo`.
3. Limpiar ruido de paginación y encabezados repetidos.
4. Separar el Código principal de `TRANSITORIOS`.
5. Extraer artículos.
6. Analizar distribución de tamaños.
7. Visualizar histogramas, ECDF y percentiles.
8. Inspeccionar los artículos más grandes.

Hipótesis inicial: **1 artículo = 1 chunk**.


In [1]:
%matplotlib inline
import sys
from pathlib import Path

# Sube hasta la carpeta que contiene shared/ y agrega este lab al path, para que
# `import chunkers` / `import compare` funcionen sin importar desde dónde se abrió Jupyter.
current_dir = Path.cwd()
labs_folder = next(d for d in [current_dir, *current_dir.parents] if (d / "shared").exists())
for import_dir in (str(labs_folder), str(labs_folder / "03_chunking")):
    if import_dir not in sys.path:
        sys.path.insert(0, import_dir)
print("labs root:", labs_folder)

labs root: /Users/savashito/Claude/Projects/AI_research/RAG/labs


In [3]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 1. Configuración

Cambia `DOCUMENT_PATH` por el nombre o ruta de tu archivo.


In [4]:
# DOCUMENT_PATH = Path("Código Nacional de Procedimientos Penales.md")
cnpp_text = (labs_folder / "ingestion" / "out" / "Sistema Penal Acusatorio"
             / "Código Nacional de Procedimientos Penales.md").read_text(encoding="utf-8")



# print(f"Archivo: {DOCUMENT_PATH}")
print(f"Caracteres: {len(cnpp_text):,}")
print(f"Palabras: {len(cnpp_text.split()):,}")
print()
print(cnpp_text[:1_500])


Caracteres: 540,734
Palabras: 82,774

**<u>CÓDIGO NACIONAL DE PROCEDIMIENTOS PENALES</u>** 

_Última Reforma DOF 28-11-2025_ 

**CÁMARA DE DIPUTADOS DEL H. CONGRESO DE LA UNIÓN** Secretaría General Secretaría de Servicios Parlamentarios 



# **CÓDIGO NACIONAL DE PROCEDIMIENTOS PENALES** 

**Nuevo Código publicado en el Diario Oficial de la Federación el 5 de marzo de 2014** 

**Última reforma publicada DOF 28-11-2025** 

Al margen un sello con el Escudo Nacional, que dice: Estados Unidos Mexicanos.- Presidencia de la República. 

**ENRIQUE PEÑA NIETO** , Presidente de los Estados Unidos Mexicanos, a sus habitantes sabed: 

Que el Honorable Congreso de la Unión, se ha servido dirigirme el siguiente 

## **DECRETO** 

**"** EL CONGRESO GENERAL DE LOS ESTADOS UNIDOS MEXICANOS, D E C R E T A : 

## **SE EXPIDE EL CÓDIGO NACIONAL DE PROCEDIMIENTOS PENALES** 

**Artículo Único.-** Se expide el Código Nacional de Procedimientos Penales. 

# **CÓDIGO NACIONAL DE PROCEDIMIENTOS PENALES** 

# *

## 2. Exploración estructural

In [5]:
patterns = {
    "LIBRO": r"(?m)^# .*LIBRO.*$",
    "TÍTULO": r"(?m)^# .*TÍTULO.*$",
    "CAPÍTULO": r"(?m)^# .*CAPÍTULO.*$",
    "SECCIÓN": r"(?m)^# .*SECCIÓN.*$",
    "ARTÍCULO": r"(?m)^## .*Artículo.*$",
}

for name, pattern in patterns.items():
    matches = re.findall(pattern, cnpp_text)
    print(f"\n{name}: {len(matches)}")
    print(*matches[:10], sep="\n")



LIBRO: 2
# **LIBRO PRIMERO DISPOSICIONES GENERALES** 
# **LIBRO SEGUNDO DEL PROCEDIMIENTO** 

TÍTULO: 19
# **TÍTULO I DISPOSICIONES PRELIMINARES** 
# **TÍTULO II PRINCIPIOS Y DERECHOS EN EL PROCEDIMIENTO** 
# **TÍTULO III COMPETENCIA** 
# **TÍTULO IV ACTOS PROCEDIMENTALES** 
# **TÍTULO V SUJETOS DEL PROCEDIMIENTO Y SUS AUXILIARES** 
# **TÍTULO VI** 
# **TÍTULO I SOLUCIONES ALTERNAS Y FORMAS DE TERMINACIÓN ANTICIPADA** 
# **TÍTULO II** 
# **TÍTULO III ETAPA DE INVESTIGACIÓN** 
# **TÍTULO IV DE LOS DATOS DE PRUEBA, MEDIOS DE PRUEBA Y PRUEBAS** 

CAPÍTULO: 60
# **CAPÍTULO ÚNICO ÁMBITO DE APLICACIÓN Y OBJETO** 
# **CAPÍTULO I PRINCIPIOS EN EL PROCEDIMIENTO** 
# **CAPÍTULO II DERECHOS EN EL PROCEDIMIENTO** 
# **CAPÍTULO I GENERALIDADES** 
# **CAPÍTULO II INCOMPETENCIA** 
# **CAPÍTULO III ACUMULACIÓN Y SEPARACIÓN DE PROCESOS** 
# **CAPÍTULO IV EXCUSAS, RECUSACIONES E IMPEDIMENTOS** 
# **CAPÍTULO I FORMALIDADES** 
# **CAPÍTULO II AUDIENCIAS** 
# **CAPÍTULO III** 

SECCIÓN: 12
# **SECCIÓN I C

## 3. Exploración del ruido

In [7]:
noise_patterns = {
    "page_numbers": r"(?m)^\d+\s+de\s+\d+\s*$",
    "code_header": r"CÓDIGO NACIONAL DE PROCEDIMIENTOS PENALES",
    "last_reform": r"Última Reforma DOF",
    "chamber_header": r"CÁMARA DE DIPUTADOS DEL H\. CONGRESO DE LA UNIÓN",
    "secretaria_general": r"Secretaría General",
    "servicios_parlamentarios": r"Secretaría de Servicios Parlamentarios",
}

for name, pattern in noise_patterns.items():
    print(f"{name:30} {len(re.findall(pattern, cnpp_text)):>5}")


page_numbers                     168
code_header                      171
last_reform                      168
chamber_header                   168
secretaria_general               170
servicios_parlamentarios         168


## 4. Limpieza

La limpieza es conservadora: elimina ruido, pero conserva contenido jurídico y estructura.


In [9]:
def is_noise_line(line: str) -> bool:
    stripped = line.strip()

    if not stripped:
        return False

    if re.fullmatch(r"\d+\s+de\s+\d+", stripped):
        return True

    if stripped in {
        "**<u>CÓDIGO NACIONAL DE PROCEDIMIENTOS PENALES</u>**",
        "CÓDIGO NACIONAL DE PROCEDIMIENTOS PENALES",
    }:
        return True

    if re.fullmatch(
        r"_?Última Reforma DOF \d{2}-\d{2}-\d{4}_?",
        stripped,
    ):
        return True

    if "CÁMARA DE DIPUTADOS DEL H. CONGRESO DE LA UNIÓN" in stripped:
        return True

    if "Secretaría General" in stripped:
        return True

    if "Secretaría de Servicios Parlamentarios" in stripped:
        return True

    return False


def clean_document(text: str) -> str:
    lines = [
        line
        for line in text.splitlines()
        if not is_noise_line(line)
    ]

    text = "\n".join(lines)

    # Quitar formato, no contenido.
    text = text.replace("**", "")
    text = text.replace("<u>", "")
    text = text.replace("</u>", "")

    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


clean_text = clean_document(cnpp_text)

print(f"Caracteres antes:   {len(cnpp_text):,}")
print(f"Caracteres después: {len(clean_text):,}")
print(f"Eliminados:         {len(cnpp_text) - len(clean_text):,}")


Caracteres antes:   540,734
Caracteres después: 498,141
Eliminados:         42,593


## 5. Inspección manual del documento limpio

In [ ]:
print(clean_text[:5_000])

## 6. Separar el Código principal de TRANSITORIOS

In [10]:
TRANSITORIOS_RE = re.compile(
    r"(?m)^#\s+(?:\*\*)?TRANSITORIOS(?:\*\*)?\s*$"
)

match = TRANSITORIOS_RE.search(clean_text)

if match:
    main_code = clean_text[:match.start()].strip()
    transitorios_and_after = clean_text[match.start():].strip()
else:
    main_code = clean_text
    transitorios_and_after = ""

print(f"Código principal: {len(main_code):,} caracteres")
print(f"Transitorios+:   {len(transitorios_and_after):,} caracteres")
print(f"Encontré TRANSITORIOS: {match is not None}")


Código principal: 424,129 caracteres
Transitorios+:   74,009 caracteres
Encontré TRANSITORIOS: True


## 7. Extraer artículos

In [ ]:
ARTICLE_RE = re.compile(
    r"(?m)^##\s+(?:\*\*)?Artículo\s+(.+?)(?:\*\*)?\s*$"
)


def article_spans(text: str):
    matches = list(ARTICLE_RE.finditer(text))

    for i, match in enumerate(matches):
        start = match.start()
        end = (
            matches[i + 1].start()
            if i + 1 < len(matches)
            else len(text)
        )

        yield (
            match.group(1).strip(),
            text[start:end].strip(),
        )


articles = list(article_spans(main_code))

print(f"Artículos encontrados: {len(articles)}")

for name, article in articles[:3]:
    print("\n" + "=" * 80)
    print(name)
    print(article[:500])


## 8. Tabla de exploración

In [ ]:
def word_count(text: str) -> int:
    return len(text.split())


article_df = pd.DataFrame({
    "index": range(1, len(articles) + 1),
    "article": [name for name, _ in articles],
    "words": [word_count(text) for _, text in articles],
    "chars": [len(text) for _, text in articles],
    "lines": [len(text.splitlines()) for _, text in articles],
})

article_df.head()

## 9. Estadísticas y percentiles

In [ ]:
percentiles = [.01, .05, .10, .25, .50, .75, .80, .90, .95, .99]

article_df["words"].describe(percentiles=percentiles)

## 10. Cobertura por límite de tamaño

In [ ]:
limits = [200, 300, 400, 500, 600, 800, 1000]

coverage = pd.DataFrame({
    "max_words": limits,
    "articles_at_or_below": [
        (article_df["words"] <= limit).sum()
        for limit in limits
    ],
    "proportion": [
        (article_df["words"] <= limit).mean()
        for limit in limits
    ],
})

coverage["proportion_percent"] = (
    coverage["proportion"] * 100
).round(2)

coverage

## 11. Los artículos más grandes

In [ ]:
article_df.nlargest(20, "words")

## 12. Histograma completo

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(article_df["words"], bins=50)
plt.xlabel("Palabras por artículo")
plt.ylabel("Número de artículos")
plt.title("Distribución de palabras por artículo")
plt.show()

## 13. Histograma 0–1000 palabras

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(article_df["words"], bins=50, range=(0, 1000))
plt.axvline(800, linestyle="--", label="800 palabras")
plt.xlabel("Palabras por artículo")
plt.ylabel("Número de artículos")
plt.title("Distribución de artículos (0–1000 palabras)")
plt.legend()
plt.show()

## 14. ECDF

Esta gráfica responde: ¿qué proporción de artículos tiene menos de X palabras?


In [ ]:
values = np.sort(article_df["words"].to_numpy())
ecdf = np.arange(1, len(values) + 1) / len(values)

plt.figure(figsize=(10, 5))
plt.plot(values, ecdf)
plt.axvline(800, linestyle="--", label="800 palabras")
plt.axhline(
    (article_df["words"] <= 800).mean(),
    linestyle="--",
)
plt.xlabel("Palabras por artículo")
plt.ylabel("Proporción acumulada")
plt.title("ECDF del tamaño de artículos")
plt.legend()
plt.show()

print(
    "Proporción ≤ 800:",
    f"{(article_df['words'] <= 800).mean():.2%}"
)

## 15. Boxplot

In [ ]:
plt.figure(figsize=(10, 3))
plt.boxplot(article_df["words"], vert=False)
plt.xlabel("Palabras por artículo")
plt.title("Boxplot del tamaño de artículos")
plt.show()

## 16. Artículos mayores a 800 palabras

In [ ]:
large_articles = article_df.query("words > 800")
print(f"Artículos > 800 palabras: {len(large_articles)}")
large_articles

## 17. Inspeccionar el artículo más grande

In [ ]:
largest_idx = article_df["words"].idxmax()
largest_name, largest_text = articles[largest_idx]

print(f"Artículo: {largest_name}")
print(f"Palabras: {article_df.loc[largest_idx, 'words']:,}")
print()
print(largest_text)

## 18. Resumen final para decidir el chunking

In [ ]:
summary = pd.Series({
    "articles": len(article_df),
    "mean_words": article_df["words"].mean(),
    "median_words": article_df["words"].median(),
    "p80_words": article_df["words"].quantile(.80),
    "p90_words": article_df["words"].quantile(.90),
    "p95_words": article_df["words"].quantile(.95),
    "p99_words": article_df["words"].quantile(.99),
    "max_words": article_df["words"].max(),
    "proportion_le_800": (article_df["words"] <= 800).mean(),
    "count_gt_800": (article_df["words"] > 800).sum(),
})

summary

# Conclusión

Si la mayoría de artículos cae bajo tu límite objetivo, usa inicialmente:

**1 artículo = 1 chunk**

Después compara:

1. Raw Article
2. Article + hierarchy
3. Recursive splitter

Y evalúa distribución, retrieval y tus métricas de RAG.
